# Pretrained models: ResNet50 trained on ImageNet
Load ResNet50 with its ImageNet weights from `keras.applications` and classify eleven photos from Wikimedia Commons
that it has never seen. No training happens in this notebook.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"          # hide TensorFlow's start-up messages
os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"  # take GPU memory only as needed
os.environ["TF_CUDNN_WORKSPACE_LIMIT_IN_MB"] = "256"   # the GPU is shared; keep scratch memory small
import warnings
warnings.filterwarnings("ignore", category=UserWarning)   # Keras notices about tf.data
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from keras import layers

tf.get_logger().setLevel("ERROR")
HERE = Path.cwd().parent if Path.cwd().name == ".logs" else Path.cwd()   # the Note folder, also from .logs/
(HERE / "data").mkdir(exist_ok=True)
print("TensorFlow", tf.__version__, "| Keras", keras.__version__, "| GPU:", bool(tf.config.list_physical_devices("GPU")))

In [ ]:
import urllib.request
PHOTOS = Path.home() / "datasets" / "photos"

def fetch(url):
    """Download a photo once (Wikimedia asks for a User-Agent) and return its local path."""
    PHOTOS.mkdir(parents=True, exist_ok=True)
    path = PHOTOS / url.split("/")[-1]
    if not path.exists():
        request = urllib.request.Request(url, headers={"User-Agent": "campusx-notes/1.0 (study notes)"})
        path.write_bytes(urllib.request.urlopen(request).read())
    return path

PICTURES = [('golden retriever', 'https://upload.wikimedia.org/wikipedia/commons/b/bd/Golden_Retriever_Dukedestiny01_drvd.jpg', 'Janneke Vreugdenhil', 'public domain'),
    ('tabby kitten', 'https://upload.wikimedia.org/wikipedia/commons/b/bb/Kittyply_edit1.jpg', 'David Corby', 'CC BY 2.5'),
    ('lion', 'https://upload.wikimedia.org/wikipedia/commons/7/73/Lion_waiting_in_Namibia.jpg', 'Kevin Pluck', 'CC BY 2.0'),
    ('zebra', 'https://upload.wikimedia.org/wikipedia/commons/8/83/Zebra_Botswana_edit02.jpg', 'Paul Maritz', 'CC BY 2.5'),
    ('African elephant', 'https://upload.wikimedia.org/wikipedia/commons/3/37/African_Bush_Elephant.jpg', 'Muhammad Mahdi Karim', 'GFDL 1.2'),
    ('lemon', 'https://upload.wikimedia.org/wikipedia/commons/e/e4/Lemon.jpg', 'Andre Karwath', 'CC BY-SA 2.5'),
    ('banana', 'https://upload.wikimedia.org/wikipedia/commons/8/8a/Banana-Single.jpg', 'Evan-Amos', 'CC BY-SA 3.0'),
    ('coffee mug', 'https://upload.wikimedia.org/wikipedia/commons/8/8d/Coffee_mug.jpg', 'Jackie Taffinder', 'CC BY-SA 3.0'),
    ('baguettes (bread)', 'https://upload.wikimedia.org/wikipedia/commons/f/f5/Baguettes_-_stonesoup.jpg', 'jules / stonesoup', 'CC BY 2.0'),
    ('hamburger', 'https://upload.wikimedia.org/wikipedia/commons/4/47/Hamburger_%28black_bg%29.jpg', 'Len Rizzi', 'public domain'),
    ('tomato', 'https://commons.wikimedia.org/wiki/Special:FilePath/Tomato_je.jpg', 'Softeis', 'CC BY-SA 3.0')]

In [ ]:
from keras.applications.resnet50 import ResNet50, preprocess_input, decode_predictions
model = ResNet50(weights="imagenet")          # downloads about 100 MB of weights the first time
print(f"ResNet50: {model.count_params():,} parameters, {len(model.layers)} Keras layers,"
      f" input {model.input_shape}, output {model.output_shape}")

In [ ]:
rows = []
OUT = HERE / "data" / "photos"
OUT.mkdir(exist_ok=True)
for i, (truth, url, author, licence) in enumerate(PICTURES):
    path = fetch(url)
    img = keras.utils.load_img(path, target_size=(224, 224))     # ResNet50 expects 224 x 224 x 3
    x = keras.utils.img_to_array(img)
    x = np.expand_dims(x, axis=0)                                 # a batch of one image
    x = preprocess_input(x)                                       # the same pixel preparation as in training
    probs = model.predict(x, verbose=0)
    top3 = decode_predictions(probs, top=3)[0]                    # [(class id, name, probability), ...]
    print(f"{truth:18s} -> " + ", ".join(f"{name} {p:.2f}" for _, name, p in top3))
    rows.append(dict(photo=truth, author=author, licence=licence,
                     **{f"top{k + 1}": top3[k][1] for k in range(3)},
                     **{f"p{k + 1}": round(float(top3[k][2]), 3) for k in range(3)}))
    img.resize((160, 160)).save(OUT / f"{i:02d}.jpg", quality=85)  # a small thumbnail for the Note's figure
pred = pd.DataFrame(rows)
pred.to_csv(HERE / "data" / "predictions.csv", index=False)

In [ ]:
# Which ImageNet classes exist? decode_predictions uses the list of the 1,000 ILSVRC classes.
import json
index = json.load(open(keras.utils.get_file("imagenet_class_index.json",
        "https://storage.googleapis.com/download.tensorflow.org/data/imagenet_class_index.json")))
names = [index[str(i)][1] for i in range(1000)]
print(len(names), "classes, e.g.", names[:5])
for word in ("tomato", "lemon", "banana", "golden_retriever", "tabby", "French_loaf", "cheeseburger", "chair", "cat"):
    print(f"{word:16s}", [n for n in names if word.lower() in n.lower()])